In [1]:
%load_ext sql
%sql sqlite:///student_data.db

In [2]:
import pandas as pd
from sqlalchemy import create_engine

# Create SQLite engine (same path used in %sql above)
engine = create_engine('sqlite:///student_data.db')

In [3]:
#1. Cleaning studentInfo.csv

# Load CSV into pandas
df_info = pd.read_csv('/Users/jamesjackson/Documents/student_outcome_analysis/data/raw/studentInfo.csv')

# Push to SQLite
df_info.to_sql('studentInfo', con=engine, index=False, if_exists='replace')

32593

In [4]:
import sqlite3

# Connect to the same SQLite database
conn = sqlite3.connect('student_data.db')

In [5]:
pd.read_sql("PRAGMA table_info(studentInfo);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,code_module,TEXT,0,None,0
1,1,code_presentation,TEXT,0,None,0
2,2,id_student,BIGINT,0,None,0
3,3,gender,TEXT,0,None,0
4,4,region,TEXT,0,None,0
5,5,highest_education,TEXT,0,None,0
6,6,imd_band,TEXT,0,None,0
7,7,age_band,TEXT,0,None,0
8,8,num_of_prev_attempts,BIGINT,0,None,0
9,9,studied_credits,BIGINT,0,None,0


In [6]:
pd.read_sql("SELECT * FROM studentInfo LIMIT 5;", conn)

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass


In [7]:
# Build SQL query to count NULLs for each column
columns = [
    "code_module", "code_presentation", "id_student", "gender", "region",
    "highest_education", "imd_band", "age_band", "num_of_prev_attempts",
    "studied_credits", "disability", "final_result"
]

null_counts = {
    col: pd.read_sql(f"SELECT COUNT(*) as nulls FROM studentInfo WHERE {col} IS NULL;", conn).iloc[0, 0]
    for col in columns
}

# Display as DataFrame
pd.DataFrame.from_dict(null_counts, orient='index', columns=['null_count'])

,null_count
code_module,0
code_presentation,0
id_student,0
gender,0
region,0
highest_education,0
imd_band,1111
age_band,0
num_of_prev_attempts,0
studied_credits,0


In [8]:
# View 20 student records where 'imd_band' is missing (NULL)
pd.read_sql("""
    SELECT * 
    FROM studentInfo 
    WHERE imd_band IS NULL 
    LIMIT 5;
""", conn)

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
0,AAA,2013J,53025,M,North Region,Post Graduate Qualification,None,55<=,0,60,N,Pass
1,AAA,2013J,71361,M,Ireland,HE Qualification,None,35-55,0,60,N,Pass
2,AAA,2013J,104476,M,Ireland,Post Graduate Qualification,None,35-55,0,60,N,Pass
3,AAA,2013J,114017,F,North Region,Post Graduate Qualification,None,35-55,0,60,N,Pass
4,AAA,2013J,172112,M,Ireland,HE Qualification,None,35-55,0,60,N,Pass


In [9]:
# Replace NULLs in 'imd_band' with 'Unknown'
cursor = conn.cursor()
cursor.execute("""
    UPDATE studentInfo 
    SET imd_band = 'Unknown' 
    WHERE imd_band IS NULL;
""")
conn.commit()

In [10]:
# Check again for NULL values
columns = [
    "code_module", "code_presentation", "id_student", "gender", "region",
    "highest_education", "imd_band", "age_band", "num_of_prev_attempts",
    "studied_credits", "disability", "final_result"
]

null_counts = {
    col: pd.read_sql(f"SELECT COUNT(*) as nulls FROM studentInfo WHERE {col} IS NULL;", conn).iloc[0, 0]
    for col in columns
}

# Display as DataFrame
pd.DataFrame.from_dict(null_counts, orient='index', columns=['null_count'])

,null_count
code_module,0
code_presentation,0
id_student,0
gender,0
region,0
highest_education,0
imd_band,0
age_band,0
num_of_prev_attempts,0
studied_credits,0


In [11]:
#2. Cleaning studentVle.csv

# Load engagement data from CSV
df_vle = pd.read_csv('/Users/jamesjackson/Documents/student_outcome_analysis/data/raw/studentVle.csv')

# Push to SQLite
df_vle.to_sql('studentVle', con=engine, index=False, if_exists='replace')

10655280

In [12]:
# Check column structure of studentVle
pd.read_sql("PRAGMA table_info(studentVle);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,code_module,TEXT,0,None,0
1,1,code_presentation,TEXT,0,None,0
2,2,id_student,BIGINT,0,None,0
3,3,id_site,BIGINT,0,None,0
4,4,date,BIGINT,0,None,0
5,5,sum_click,BIGINT,0,None,0


In [13]:
# Check for NULL values in each column of the studentVle table
vle_columns = ["code_module", "code_presentation", "id_student", "id_site", "date", "sum_click"]

vle_null_counts = {
    col: pd.read_sql(f"SELECT COUNT(*) AS nulls FROM studentVle WHERE {col} IS NULL;", conn).iloc[0, 0]
    for col in vle_columns
}

# Display as DataFrame
pd.DataFrame.from_dict(vle_null_counts, orient="index", columns=["null_count"])

,null_count
code_module,0
code_presentation,0
id_student,0
id_site,0
date,0
sum_click,0


In [15]:
#3. Cleaning studentAssessment.csv

# Load engagement data from CSV
df_vle = pd.read_csv('/Users/jamesjackson/Documents/student_outcome_analysis/data/raw/studentAssessment.csv')

# Push to SQLite
df_vle.to_sql('studentAssessment', con=engine, index=False, if_exists='replace')

173912

In [16]:
# Check column structure of studentVle
pd.read_sql("PRAGMA table_info(studentAssessment);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,id_assessment,BIGINT,0,None,0
1,1,id_student,BIGINT,0,None,0
2,2,date_submitted,BIGINT,0,None,0
3,3,is_banked,BIGINT,0,None,0
4,4,score,FLOAT,0,None,0


In [17]:
# Check for NULLs in studentAssessment
assessment_columns = ["id_assessment", "id_student", "date_submitted", "is_banked", "score"]

assessment_null_counts = {
    col: pd.read_sql(f"SELECT COUNT(*) AS nulls FROM studentAssessment WHERE {col} IS NULL;", conn).iloc[0, 0]
    for col in assessment_columns
}

pd.DataFrame.from_dict(assessment_null_counts, orient="index", columns=["null_count"])

,null_count
id_assessment,0
id_student,0
date_submitted,0
is_banked,0
score,173


In [18]:
# View 20 student assessments where score is missing
pd.read_sql("""
    SELECT * 
    FROM studentAssessment 
    WHERE score IS NULL 
    LIMIT 20;
""", conn)

,id_assessment,id_student,date_submitted,is_banked,score
0,1752,721259,22,0,None
1,1754,260355,127,0,None
2,1760,2606802,180,0,None
3,14984,186780,77,0,None
4,14984,531205,26,0,None
5,14984,534151,7,0,None
6,14984,549713,84,0,None
7,14984,554393,55,0,None
8,14985,186780,77,0,None
9,14986,33666,117,0,None


In [19]:
# Delete rows with NULL score from studentAssessment
cursor = conn.cursor()
cursor.execute("""
    DELETE FROM studentAssessment 
    WHERE score IS NULL;
""")
conn.commit()

In [20]:
# Count how many rows have negative submission dates
pd.read_sql("""
    SELECT COUNT(*) AS negative_dates
    FROM studentAssessment
    WHERE date_submitted = -1;
""", conn)

,negative_dates
0,1926


In [21]:
# Set all negative submission dates to NULL (clean up unreliable entries)
# 94% of negative dates were -1, suggesting use as placeholder / system default etc.
cursor.execute("""
    UPDATE studentAssessment
    SET date_submitted = NULL
    WHERE date_submitted < 0;
""")
conn.commit()

In [22]:
#4. Cleaning assessments.csv

# Load engagement data from CSV
df_vle = pd.read_csv('/Users/jamesjackson/Documents/student_outcome_analysis/data/raw/assessments.csv')

# Push to SQLite
df_vle.to_sql('assessments', con=engine, index=False, if_exists='replace')

206

In [23]:
# Check column structure of studentVle
pd.read_sql("PRAGMA table_info(studentAssessment);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,id_assessment,BIGINT,0,None,0
1,1,id_student,BIGINT,0,None,0
2,2,date_submitted,BIGINT,0,None,0
3,3,is_banked,BIGINT,0,None,0
4,4,score,FLOAT,0,None,0


In [24]:
# Check for NULL values in the assessments table
assessment_meta_cols = [
    "id_assessment", "code_module", "code_presentation", 
    "assessment_type", "date", "weight"
]

meta_null_counts = {
    col: pd.read_sql(f"SELECT COUNT(*) AS nulls FROM assessments WHERE {col} IS NULL;", conn).iloc[0, 0]
    for col in assessment_meta_cols
}

pd.DataFrame.from_dict(meta_null_counts, orient="index", columns=["null_count"])


,null_count
id_assessment,0
code_module,0
code_presentation,0
assessment_type,0
date,11
weight,0


In [25]:
# View all assessments where 'date' is NULL
pd.read_sql("""
    SELECT * 
    FROM assessments 
    WHERE date IS NULL;
""", conn)


,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1757,Exam,None,100.0
1,AAA,2014J,1763,Exam,None,100.0
2,BBB,2013B,14990,Exam,None,100.0
3,BBB,2013J,15002,Exam,None,100.0
4,BBB,2014B,15014,Exam,None,100.0
5,BBB,2014J,15025,Exam,None,100.0
6,CCC,2014B,24290,Exam,None,100.0
7,CCC,2014B,40087,Exam,None,100.0
8,CCC,2014J,24299,Exam,None,100.0
9,CCC,2014J,40088,Exam,None,100.0


In [26]:
# View first 5 rows from assessments table
pd.read_sql("""
    SELECT * 
    FROM assessments 
    WHERE date IS NOT NULL
    LIMIT 5;
""", conn)

,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1752,TMA,19.0,10.0
1,AAA,2013J,1753,TMA,54.0,20.0
2,AAA,2013J,1754,TMA,117.0,20.0
3,AAA,2013J,1755,TMA,166.0,20.0
4,AAA,2013J,1756,TMA,215.0,30.0


In [27]:
# Create a table named 'assessment_scores_joined' with student assessment scores + module info
cursor.execute("""
    CREATE TABLE assessment_scores_joined AS
    SELECT 
        sa.id_student,
        a.code_module,
        a.code_presentation,
        sa.score
    FROM studentAssessment sa
    JOIN assessments a 
        ON sa.id_assessment = a.id_assessment
    WHERE sa.score IS NOT NULL;
""")
conn.commit()

# Selected only id_student, code_module, code_presentation, and score
# because these are the essential fields needed to calculate per-student
# assessment performance for each module presentation

OperationalError: table assessment_scores_joined already exists

In [28]:
# Preview 5 rows from the new joined table
pd.read_sql("""
    SELECT * 
    FROM assessment_scores_joined 
    LIMIT 5;
""", conn)

,id_student,code_module,code_presentation,score
0,11391,AAA,2013J,78.0
1,28400,AAA,2013J,70.0
2,31604,AAA,2013J,72.0
3,32885,AAA,2013J,69.0
4,38053,AAA,2013J,79.0


In [29]:
# Create a new table with average score and number of assessments per student-module-presentation
cursor.execute("""
    CREATE TABLE assessment_scores_aggregated AS
    SELECT 
        id_student,
        code_module,
        code_presentation,
        AVG(score) AS avg_score,
        COUNT(score) AS num_assessments
    FROM assessment_scores_joined
    GROUP BY id_student, code_module, code_presentation;
""")
conn.commit()


OperationalError: table assessment_scores_aggregated already exists

In [34]:
# Preview 5 rows from the aggregated assessment feature table
pd.read_sql("""
    SELECT * 
    FROM assessment_scores_aggregated 
    LIMIT 5;
""", conn)

,id_student,code_module,code_presentation,avg_score,num_assessments
0,6516,AAA,2014J,61.800000,5
1,8462,DDD,2013J,87.666667,3
2,8462,DDD,2014J,86.500000,4
3,11391,AAA,2013J,82.000000,5
4,23629,BBB,2013B,82.500000,4


In [35]:
# Create engagement_aggregated table: total clicks per student per module presentation
cursor.execute("""
    CREATE TABLE engagement_aggregated AS
    SELECT 
        id_student,
        code_module,
        code_presentation,
        SUM(sum_click) AS total_clicks
    FROM studentVle
    GROUP BY id_student, code_module, code_presentation;
""")
conn.commit()


In [36]:
# Preview 5 rows from the aggregated engagement feature table
pd.read_sql("""
    SELECT * 
    FROM engagement_aggregated 
    LIMIT 5;
""", conn)

,id_student,code_module,code_presentation,total_clicks
0,6516,AAA,2014J,2791
1,8462,DDD,2013J,646
2,8462,DDD,2014J,10
3,11391,AAA,2013J,934
4,23629,BBB,2013B,161


In [30]:
# studentInfo: baseline number of student-module-presentation rows
pd.read_sql("SELECT COUNT(*) AS rows FROM studentInfo;", conn)


,rows
0,32593


In [31]:
# assessment_scores_aggregated: should be fewer (not all students submitted work)
pd.read_sql("SELECT COUNT(*) AS rows FROM assessment_scores_aggregated;", conn)


,rows
0,25820


In [32]:
# engagement_aggregated: same — some students didn’t click on anything
pd.read_sql("SELECT COUNT(*) AS rows FROM engagement_aggregated;", conn)


,rows
0,29228


In [40]:
# Join all features into one master table per student-module-presentation
cursor.execute("""
    CREATE TABLE student_master AS
    SELECT 
        si.id_student,
        si.code_module,
        si.code_presentation,
        si.gender,
        si.region,
        si.highest_education,
        si.imd_band,
        si.age_band,
        si.num_of_prev_attempts,
        si.studied_credits,
        si.disability,
        si.final_result,
        asa.avg_score,
        asa.num_assessments,
        ea.total_clicks
    FROM studentInfo si
    LEFT JOIN assessment_scores_aggregated asa
        ON si.id_student = asa.id_student
        AND si.code_module = asa.code_module
        AND si.code_presentation = asa.code_presentation
    LEFT JOIN engagement_aggregated ea
        ON si.id_student = ea.id_student
        AND si.code_module = ea.code_module
        AND si.code_presentation = ea.code_presentation;
""")
conn.commit()


In [33]:
# Preview 5 rows from the final master table
pd.read_sql("""
    SELECT * 
    FROM student_master 
    LIMIT 5;
""", conn)

,id_student,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,avg_score,num_assessments,total_clicks
0,11391,AAA,2013J,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,82.0,5.0,934
1,28400,AAA,2013J,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,66.4,5.0,1435
2,30268,AAA,2013J,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,NaN,NaN,281
3,31604,AAA,2013J,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass,76.0,5.0,2158
4,32885,AAA,2013J,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass,54.4,5.0,1034


In [34]:
# Check for nulls in each column of the master table
pd.read_sql("""
    SELECT 
        SUM(CASE WHEN id_student IS NULL THEN 1 ELSE 0 END) AS id_student,
        SUM(CASE WHEN code_module IS NULL THEN 1 ELSE 0 END) AS code_module,
        SUM(CASE WHEN code_presentation IS NULL THEN 1 ELSE 0 END) AS code_presentation,
        SUM(CASE WHEN gender IS NULL THEN 1 ELSE 0 END) AS gender,
        SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END) AS region,
        SUM(CASE WHEN highest_education IS NULL THEN 1 ELSE 0 END) AS highest_education,
        SUM(CASE WHEN imd_band IS NULL THEN 1 ELSE 0 END) AS imd_band,
        SUM(CASE WHEN age_band IS NULL THEN 1 ELSE 0 END) AS age_band,
        SUM(CASE WHEN num_of_prev_attempts IS NULL THEN 1 ELSE 0 END) AS num_of_prev_attempts,
        SUM(CASE WHEN studied_credits IS NULL THEN 1 ELSE 0 END) AS studied_credits,
        SUM(CASE WHEN disability IS NULL THEN 1 ELSE 0 END) AS disability,
        SUM(CASE WHEN final_result IS NULL THEN 1 ELSE 0 END) AS final_result,
        SUM(CASE WHEN avg_score IS NULL THEN 1 ELSE 0 END) AS avg_score,
        SUM(CASE WHEN num_assessments IS NULL THEN 1 ELSE 0 END) AS num_assessments,
        SUM(CASE WHEN total_clicks IS NULL THEN 1 ELSE 0 END) AS total_clicks
    FROM student_master;
""", conn)

,id_student,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,avg_score,num_assessments,total_clicks
0,0,0,0,0,0,0,0,0,0,0,0,0,6773,6773,3365


In [38]:
# Count how many students are missing withdrew
pd.read_sql("""
    SELECT COUNT(*) AS num_withdrawn
    FROM student_master
    WHERE final_result = 'Withdrawn';
""", conn)

,num_withdrawn
0,10156


In [42]:
# Count how many students are missing assessment data (avg_score or num_assessments)
# even though they did NOT withdraw — helps identify unexpected gaps in data
pd.read_sql("""
    SELECT COUNT(*) AS missing_but_not_withdrawn
    FROM student_master
    WHERE final_result != 'Withdrawn'
      AND (avg_score IS NULL AND num_assessments IS NULL);
""", conn)

,missing_but_not_withdrawn
0,1290


In [43]:
# 1290 students had NULL avg_score / num_assessments despite not withdrawing

In [44]:
# Count how many students have missing total_clicks
# even though they did NOT withdraw — helps detect unexpected inactivity
pd.read_sql("""
    SELECT COUNT(*) AS missing_clicks_but_not_withdrawn
    FROM student_master
    WHERE final_result != 'Withdrawn'
      AND total_clicks IS NULL;
""", conn)

,missing_clicks_but_not_withdrawn
0,377


In [45]:
# 377 students had NULL total_clicks despite not withdrawing

In [47]:
# Check for duplicate student-module-presentation combinations
pd.read_sql("""
    SELECT COUNT(*) - COUNT(DISTINCT id_student || code_module || code_presentation) AS num_duplicates
    FROM student_master;
""", conn)

,num_duplicates
0,0


In [48]:
# Check for leading/trailing whitespace in all text columns
text_columns = [
    'gender', 'region', 'highest_education',
    'imd_band', 'age_band', 'disability', 'final_result'
]

# Build a dictionary of counts where trimmed value differs from original
whitespace_issues = {
    col: pd.read_sql(f"""
        SELECT COUNT(*) AS count 
        FROM student_master 
        WHERE TRIM({col}) != {col}
    """, conn).iloc[0, 0]
    for col in text_columns
}

# Show columns with any issues
pd.DataFrame.from_dict(whitespace_issues, orient='index', columns=['whitespace_count'])


,whitespace_count
gender,0
region,0
highest_education,0
imd_band,0
age_band,0
disability,0
final_result,0


In [49]:
# Load the final cleaned table from SQLite
df_master = pd.read_sql("SELECT * FROM student_master;", conn)

# Export to CSV
df_master.to_csv('/Users/jamesjackson/Documents/student_outcome_analysis/data/master-table.csv', index=False)